In [ ]:
from IPython.display import HTML, display
output_struktur = """
<style>
.program-card {
    font-family: Arial, sans-serif;
    max-width: 700px}


.program-title {
    background: #eef5f1;
    color: #2f5d50;
    padding: 10px;
    border-radius: 8px;
    font-size: 20px;
    margin-bottom: 8px; /* weniger Abstand nach Titel */
    text-align: left;
    font-weight: 600}

details {
    background: #fafbfb;
    border-radius: 8px;
    padding: 10px;
    margin-bottom: 4px; /* sehr kleiner Abstand zwischen Boxen */
    border: 1px solid #e6ebe9;
    transition: 0.2s ease;}

summary {
    font-weight: 600;
    cursor: pointer;
    padding: 10px;
    font-size: 15px;
    color: #2f5d50}

.subitem {
    margin-left: 18px;
    color: #555;
    font-size: 14px;
    line-height: 1.4; /* kompaktere Zeilen */}
</style>

<div class="program-card">

<div class="program-title">
Programmstruktur
</div>

<details open>
<summary>1. Wärmeleistung der Anlage</summary>
<div class="subitem">
1.1 Spitzenbelastung als Auslegungswert<br>
(GW-Entzugsleistung und Förderrate)
</div>
</details>

<details open>
<summary>2. Hydrogeologische Berechnungen</summary>
<div class="subitem">
2.1 Hydrogeologische Eingangsparameter<br>
2.2 Fassungsvermögen Qf und Wasserandrang Qa<br>
2.3 Optimale Brunnenleistung Qopt
</div>
</details>

<details open>
<summary>3. Brunnendimensionierung</summary>
<div class="subitem">
3.1 Minimaler Brunnenabstand<br>
3.2 Rückströmung
</div>
</details>

<details open>
<summary>4. Finale Auswertung</summary>
</details>
</div>
"""
display(HTML(output_struktur))


In [ ]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
import sympy as sp
import math

from matplotlib import pyplot as plt
from IPython.display import display, HTML
from scipy.interpolate import interp1d
from scipy.optimize import brentq

# 1. Wärmeleistung der Anlage

## 1.1. Spitzenbelastung als Auslegungswert

In [ ]:
label_layout = widgets.Layout(width='150px')
input_layout = widgets.Layout(width='300px')

# URL
URL_label = widgets.HTML("<b>URL Klimadiagramm:</b>", layout=label_layout)
URL_eingabe = widgets.Text(
    value="",
    placeholder="https://www.klimadiagramme.de/...",
    layout=input_layout)

URL_zeile = widgets.HBox([URL_label, URL_eingabe])

# Ort
ort_label = widgets.HTML("<b>Gewählter Ort:</b>", layout=label_layout)
ort_eingabe = widgets.Text(
    value="",
    placeholder="z.B. Aachen",
    layout=input_layout)

ort_zeile = widgets.HBox([ort_label, ort_eingabe])

# Gesamter Output
output_ort = widgets.VBox(
    [URL_zeile, ort_zeile],
    layout=widgets.Layout(
        padding='15px',
        border='1px solid lightgray',
        width='500px'))

display(output_ort)


In [ ]:
# Klimagraph für Städte/Regionen in Deutschland
URL = URL_eingabe.value
Ort = ort_eingabe.value
tables = pd.read_html(URL) 

climatediag_df = tables[1]
climatediag_df = climatediag_df.iloc[5:].reset_index(drop=True)
climatediag_df.columns = ['Monat', 'Niederschlag [mm]', 'Temperatur [°C]']
climatediag_df = climatediag_df.astype({"Temperatur [°C]": float, "Niederschlag [mm]": float})

In [ ]:
clim_df_plot = climatediag_df[climatediag_df["Monat"] != "Jahr"]
clim_x = clim_df_plot["Monat"]
clim_y1 = clim_df_plot["Temperatur [°C]"]
clim_y2 = clim_df_plot["Niederschlag [mm]"]

fig, ax1 = plt.subplots(figsize=(6,4))
plt.title("Klimagraph für " + Ort)

# Linke y-Achse für Niederschlag
ax1.plot(clim_x, clim_y2, color='blue', linewidth=1, label="Niederschlag (mm)")
ax1.fill_between(clim_x, clim_y2, color='blue', alpha=0.2)
ax1.bar(clim_x, clim_y2, alpha=0.3, color='blue', width=0.1)
ax1.axhline(0, color='black', linewidth=0.5)
ax1.set_ylabel("Niederschlag (mm)", color='blue')
ax1.set_xlabel("Monat")
ax1.set_xticks(clim_x)
ax1.set_xlim([0, 11])

# Rechte y-Achse für Temperatur
ax2 = ax1.twinx()
ax2.plot(clim_x, clim_y1, linewidth=2, color='red', label="Temperatur (°C)")
ax2.set_ylabel('Temperatur (°C)', color='red')  

# Anpassung der y-Achsen, damit 0 auf beiden Achsen übereinstimmt
y1_min = -10
y1_max = clim_df_plot["Niederschlag [mm]"].max() + 10
ax1.set_ylim(y1_min, y1_max)
zero_pos = (0 - y1_min) / (y1_max - y1_min)
temp_max = clim_df_plot["Temperatur [°C]"].max() + 5
temp_min = -zero_pos / (1 - zero_pos) * temp_max
ax2.set_ylim(temp_min, temp_max)


fig.tight_layout()
fig.legend(bbox_to_anchor=(0.57, 0.92), loc='upper left')
plt.close(fig)

output_klimadiag = widgets.Output()
with output_klimadiag:
    display(fig)

output_klimadf = widgets.Output()
with output_klimadf:
    display(climatediag_df)

display(widgets.HBox([output_klimadf, output_klimadiag])) # Anzeige von DataFrame und Klimagraph nebeneinander

In [ ]:
title = widgets.HTML(value="""<div style='font-size:22px; font-weight:600; margin-bottom:20px; '>Gebäudekonfiguration</div>""")

# Linke Spalte mit Gebädetyp und Gebäudefläche
label_typ = widgets.HTML("<b>Gebäudetyp</div>")
Geb_typ = widgets.Dropdown(
    options=['Einfamilienhaus', 'Mehrfamilienhaus', 'Schule'],
    value='Einfamilienhaus',
    layout=widgets.Layout(width='320px'))

label_fläche = widgets.HTML("<b>Fläche (m²)</b>")
Geb_fläche = widgets.FloatText(
    value=100,
    layout=widgets.Layout(width='320px'))

left_column = widgets.VBox(
    [label_typ, Geb_typ, label_fläche, Geb_fläche],
    layout=widgets.Layout(gap='20px'))

# Rechte Spalte mit Betriebsart
label_betrieb = widgets.HTML("<div style='font-weight:500;'<b>Betriebsart</div>")
betrieb = widgets.SelectMultiple(
    options=['Heizen', 'Kühlen'],
    value=['Heizen'],
    layout=widgets.Layout(width='340px', height='92px'))

hint = widgets.HTML("<div style='font-size:12px; color:gray;'>Strg/Shift für Mehrfachauswahl</div>")

right_column = widgets.VBox(
    [label_betrieb, betrieb, hint],
    layout=widgets.Layout(gap='8px'))

# Zusammengefasst
geb_konfig = widgets.HBox(
    [left_column, right_column],
    layout=widgets.Layout(gap='50px', align_items='flex-start'))

# Output Arrangement
output_gebäude_konfig = widgets.VBox(
    [title, geb_konfig],
    layout=widgets.Layout(padding='10px', border_radius='14px', width='700px', background_color='white'))

display(output_gebäude_konfig)

In [ ]:
title = widgets.HTML(
    value="""<div style='font-size:22px; font-weight:600;'>Angaben für Betrieb und Wärmepumpe</div>""",
    layout=widgets.Layout(margin='0px 0px 0px 0px'))

# Linke Spalte für Heizbetrieb
label_heizbetrieb = widgets.HTML("<b>Heizbetrieb</b>")

label_spez_heizlast = widgets.HTML("Spezifische Heizlast [W/m²]")
spez_heizlast = widgets.FloatText(
    value=70,
    layout=widgets.Layout(width='200px'))

label_COP_heizen = widgets.HTML("COP Heizen")
COP_heizen = widgets.FloatText(
    value=4,
    layout=widgets.Layout(width='200px'))

left_column = widgets.VBox(
    [label_heizbetrieb, label_spez_heizlast, spez_heizlast, label_COP_heizen, COP_heizen],
    layout=widgets.Layout(gap='6px'))

# Rechte Spalte für Kühlbetrieb
label_kühlbetrieb = widgets.HTML("<b>Kühlbetrieb</b>")

label_spez_kühllast = widgets.HTML("Spezifische Kühllast [W/m²]")
spez_kühllast = widgets.FloatText(
    value=40,
    layout=widgets.Layout(width='200px'))

label_COP_kühlen = widgets.HTML("COP Kühlen")
COP_kühlen = widgets.FloatText(
    value=4,
    layout=widgets.Layout(width='200px'))

right_column = widgets.VBox(
    [label_kühlbetrieb, label_spez_kühllast, spez_kühllast, label_COP_kühlen, COP_kühlen],
    layout=widgets.Layout(gap='5px'))

left_column.layout.width = '250px'
right_column.layout.width = '250px'

# Richtwerte Tabellen
Geb_richtwerte_heiz = pd.DataFrame({
    "Gebäude Baujahr": ["Altbau (vor 1977)", "Sanierter Altbau (1978-1995)", "Neubau (ab 2002)"],
    "Spezifische Heizlast [W/m²]": ["120 - 95", "95 - 70", "50 - 25"] # Gilt für Normaußenbedingungen
    })

Geb_richtwerte_kühl = pd.DataFrame({
    "Gebäudetyp": ["Wohngebäude", "Schule"],
    "Spezifische Kühllast [W/m²]": ["30 - 45", "40 - 55"]})

table_output_heiz = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #E5E7EB',
        padding='6px',
        width='420px',
        max_height='160px',
        overflow='auto'))

table_output_kühl = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #E5E7EB',
        padding='6px',
        width='420px',
        max_height='120px',
        overflow='auto'))

with table_output_heiz:
    display(Geb_richtwerte_heiz)

with table_output_kühl:
    display(Geb_richtwerte_kühl)
    
table_output_heiz.layout.width = '400px'
table_output_kühl.layout.width = '300px'

tables = widgets.HBox(
    [table_output_heiz, table_output_kühl],
    layout=widgets.Layout(gap='50px'))

# Zusammengefasst
if "Kühlen" in betrieb.value:
    HK_last = widgets.HBox(
        [left_column, right_column, tables],
        layout=widgets.Layout(gap='5px', align_items='center'))
else:
    HK_last = widgets.HBox(
        [left_column, tables],
        layout=widgets.Layout(gap='5px', align_items='center'))

# Output Arrangement
output_last = widgets.VBox(
    [title, HK_last],
    layout=widgets.Layout(padding='10px', border_radius='14px', width='1300px', background_color='white'))

display(output_last)

In [ ]:
output_GW_temp = widgets.Output()

# Berechnung der Grundwassertemperatur
GW_temp = climatediag_df['Temperatur [°C]'][:12].mean()
# Grenzwerte bestimmen
if GW_temp - 6 < 5:
    temp_min = 5
else:
    temp_min = GW_temp - 6

if GW_temp + 6 > 20:
    temp_max = 20
else:
    temp_max = GW_temp + 6

# Slider zur Festlegung der Temperaturspreizung
T_slider = widgets.IntSlider(
    value=5,
    min=0,
    max=6,
    step=1,
    description='Temperaturdifferenz ΔT [K]:',
    continuous_update=False,
    layout=widgets.Layout(width='460px'),
    style={'description_width': '210px'}
)

# Erklärung: Erlaubte Veränderung der Grundwassertemperatur
with output_GW_temp:
    display(HTML(f"""
    <h3 style='color:cornflowerblue'>Veränderung der Grundwassertemperatur</h3>
    <div style="
        background:whitesmoke;
        padding:15px;
        border-radius:12px;
        width:600px;
        font-size:14px;
        line-height:1.6">
        
        <b>Die lokale Grundwassertemperatur beträgt ca.</b>
        <span style="color:#2c7be5">
            {round(GW_temp,1)} °C 
        </span>
        (Jahresdurchschnitt der Lufttemperatur)
        <br><br>

        <b>Die Temperaturspreizung ΔT des Grundwassers darf maximal 6 K betragen.</b><br>
        <b>Minimale Einleittemperatur:</b> 5 °C<br>
        <b>Maximale Einleittemperatur:</b> 20 °C (Kühlbetrieb)<br>
        <hr>
        <b>Zulässiger Bereich:</b>
        <span style="color:#2c7be5">
        {round(temp_min,1)} °C – {round(temp_max,1)} °C
        </span>
    </div>
    """))

# Alles anzeigen
display(output_GW_temp)
display(T_slider)


In [ ]:
# Berechnung der maximalen Heiz- und Kühlleistung
output_max_leistung = widgets.Output()
P_h_max = round(Geb_fläche.value * spez_heizlast.value, 1) # W
P_gw_h_max = round(P_h_max * (1-(1/COP_heizen.value)), 1)

if "Kühlen" in betrieb.value:
    P_k_max = round(Geb_fläche.value * spez_kühllast.value, 1) # W
    P_gw_k_max = round(P_k_max * (1-(1/COP_kühlen.value)), 1)


if "Kühlen" in betrieb.value:
    with output_max_leistung:
        display(HTML(f"""
        <h3 style='color:cornflowerblue'>Berechnung der maximalen Heiz- und Kühlleistung</h3>
        <div style="
            background:whitesmoke;
            padding:15px;
            border-radius:12px;
            width:580px;
            font-size:14px;
            line-height:1.6">
            
            Die maximale Heizleistung beträgt</>
            <span style="color:black">
                {P_h_max} W
            </span>
            <br>
            Die maximale Kühlleistung beträgt</>
            <span style="color:black">
                {P_k_max} W
            </span>
            <hr>
            Die maximale GW-Entzugsleistung für Heizung beträgt</>
            <span style="color:black">
                {P_gw_h_max} W
            </span>
            <br>
            Die maximale GW-Entzugsleistung für Kühlung beträgt</>
            <span style="color:black">
                {P_gw_k_max} W
            </span>        
        </div>
        """))
else:
    with output_max_leistung:
        display(HTML(f"""
        <h3 style='color:cornflowerblue'>Berechnung der maximalen Heiz- und Kühlleistung</h3>
        <div style="
            background:whitesmoke;
            padding:15px;
            border-radius:12px;
            width:580px;
            font-size:14px;
            line-height:1.6">
            
            Die maximale Heizleistung beträgt</>
            <span style="color:black">
                {P_h_max} W
            </span>
            <hr>
            Die maximale GW-Entzugsleistung für Heizung beträgt</>
            <span style="color:black">
                {P_gw_h_max} W
            </span>   
        </div>
        """))
display(output_max_leistung)

In [ ]:
def calc_Q_vol(P_gw, delta_T):
    """Berechnung des benötigten Fördervolumenstroms in m³/h.
    Args:
        P_gw (float): Entzugsleistung des Grundwassers in W
        delta_T (float): Temperaturdifferenz in K"""
    rho = 1000  # Dichte in kg/m³
    c = 4180  # Spezifische Wärmekapazität in J/kgK
    Q_vol = (P_gw) / (rho * c * delta_T) #W/(J/m3)--> m3/s
    Q_vol_m3h = Q_vol * 3600  # Umrechnung von m³/s auf m³/h
    Q_vol_Ls = Q_vol * 1000 # Umrechnung von m³/s auf L/s
    return round(Q_vol, 5), round(Q_vol_m3h, 5), round(Q_vol_Ls, 5)

Q_vol_h_value = calc_Q_vol(P_gw_h_max, T_slider.value)
if "Kühlen" in betrieb.value:
    Q_vol_k_value = calc_Q_vol(P_gw_k_max, T_slider.value)

output_GW_vol = widgets.Output()

if "Kühlen" in betrieb.value:
    with output_GW_vol:
        if "Kühlen" in betrieb.value:
            max_value = round(max(Q_vol_h_value[1], Q_vol_k_value[1]), 2)
        else:
            max_value = round(Q_vol_h_value[1], 2)
        display(HTML(f"""
        <h3 style='color:cornflowerblue'>Berechnung der erforderlichen Grundwasser-Förderrate</h3>
        <div style="
            background:whitesmoke;
            padding:15px;
            border-radius:12px;
            width:450px;
            font-size:14px;
            line-height:1.6">
            
            Die Grundwasser-Förderrate für den Heizbetrieb:</>
            <div style="color:black">
                {Q_vol_h_value[0]} m³/s<br>
                {Q_vol_h_value[1]} m³/h<br>
                {Q_vol_h_value[2]} L/s
            </div>
            <br>
            Die Grundwasser-Förderrate für den Kühlbetrieb:</>
            <div style="color:black">
                {Q_vol_k_value[0]} m³/s<br>
                {Q_vol_k_value[1]} m³/h<br>
                {Q_vol_k_value[2]} L/s
            </div>  
            <br>
            <b>Die maximal erforderliche GW-Förderrate beträgt:</b>
            <span style="color:black">
               <b>{round(max_value,2)}</b> <b>m³/h</b>
            </span>
        </div>
        """))

else:
    with output_GW_vol:
        display(HTML(f"""
        <h3 style='color:cornflowerblue'>Berechnung der erforderlichen Grundwasser-Förderrate</h3>
        <div style="
            background:whitesmoke;
            padding:15px;
            border-radius:12px;
            width:450px;
            font-size:14px;
            line-height:1.8">

            Die Grundwasser-Förderrate für den Heizbetrieb:<br>

            <div style="color:black">
                {Q_vol_h_value[0]} m³/s<br>
                {Q_vol_h_value[1]} m³/h<br>
                {Q_vol_h_value[2]} L/s
            </div>
            <br>
            <b>Die maximal erforderliche GW-Förderrate beträgt:</b>
            <span style="color:black">
                <b>{round(Q_vol_h_value[1],2)}</b> <b>m³/h</b>
            </span>

        </div>
        """))

display(output_GW_vol)


# 2. Hydrogeologische Berechnungen

## 2.1. Hydrogeologische Eingangsparameter

In [ ]:
# Dropdown-Menu für Brunnendurchmesser nach DIN EN ISO 6708
d_br_dropdown = widgets.Dropdown(
    options=[80, 100, 125, 150, 200, 250, 300, 350],
    value=80, # Standardwert
    description='Wähle Brunnendurchmesser [mm]:',
    disabled=False,
    layout=widgets.Layout(width='300px'),
    style={'description_width': '200px'})
display(d_br_dropdown)


In [ ]:
d_bl_calc = d_br_dropdown.value # Brunnendurchmesser = Endbohrdurchmesser (Bohrbrunnen Bieske 1998)
r_bl_calc = d_bl_calc/2 # in mm
r_br_calc = d_br_dropdown.value/2 # in mm

In [ ]:
# Eingabefelder hydrogeologischer Parameter
width_input2 = '280px'
style_input2 = {'description_width': '180px'}

r_br_wert = widgets.FloatText(
    value=r_br_calc/1000,
    description='Brunnenradius [m]:',
    layout=widgets.Layout(width=width_input2),
    style=style_input2)

r_bl_wert = widgets.FloatText(
    value=r_bl_calc/1000,
    description='Bohrlochradius [m]:',
    layout=widgets.Layout(width=width_input2),
    style=style_input2)

kf_wert = widgets.FloatText(
    value=10**-4,
    description='kf-Wert [m/s]:',
    layout=widgets.Layout(width=width_input2),
    style=style_input2)

kf_fehler = widgets.FloatText( # kf_fehler.value ist die Breite des log10-Unsicherheitsintervalls, Bsp: 1 -> kf liegt zwischen 0.1*kf und 10*kf
    value=0.5,
    description='kf-Wert Fehler: log(kf) =',
    layout=widgets.Layout(width=width_input2),
    style=style_input2) 

hf_wert = widgets.FloatText(
    value=20,
    description='Filterlänge [m]:',
    layout=widgets.Layout(width=width_input2),
    style=style_input2)

hf_fehler = widgets.FloatText(
    value=1,
    description='Filterlänge Fehler [m]:',
    layout=widgets.Layout(width=width_input2),
    style=style_input2)

i_wert = widgets.FloatText(
    value=0.02,
    description='Hydraul. Gradient [-]:',
    layout=widgets.Layout(width=width_input2),
    style=style_input2)

output_hydro_params = widgets.VBox([
    widgets.HTML("""
    <div style="
        background:white;
        color:cornflowerblue;
        padding:10px 15px;
        border-radius:10px 10px 0 0;
        font-size:16px;
        font-weight:bold;
        width:290px;
        display:inline-block;">
        Parameter & Unsicherheiten
    </div>
    """),

    widgets.VBox([r_br_wert, r_bl_wert, kf_wert, kf_fehler, hf_wert, hf_fehler, i_wert], 
        layout=widgets.Layout(padding='15px', border='1px solid #eee', border_radius='0 0 12px 12px', background='whitesmoke', width='305px'))
])

display(output_hydro_params)

## 2.2. Fassungsvermögen Qf und Wasserandrang Qa

In [ ]:
# Berechnung Qa, Qf und Fehler

# Absenkung Datenreihe zur Visualisierung
s_array = np.arange(0.1, 0.9*hf_wert.value, 0.05) # in m
# Berechnung des effektiven Brunnenradius für Qa Formel
calc_r_e=round((r_bl_wert.value + r_br_wert.value)/2, 3) #in m

# Symbole definieren
kf, hF, s_sym, r_eff = sp.symbols('kf hF s r_eff')
R = 3000 * s_sym * sp.sqrt(kf)
Q_a_expr = sp.pi * kf * ((hF**2 - (hF - s_sym)**2) / sp.log(R / r_eff))

# Ableitungen
dQ_dkf_expr = sp.diff(Q_a_expr, kf)
dQ_dhF_expr = sp.diff(Q_a_expr, hF)

Qa_func = sp.lambdify((s_sym, kf, hF, r_eff), Q_a_expr, "numpy")
dQ_dkf_func = sp.lambdify((s_sym, kf, hF, r_eff), dQ_dkf_expr, "numpy")
dQ_dhF_func = sp.lambdify((s_sym, kf, hF, r_eff), dQ_dhF_expr, "numpy")
delta_hF = hf_fehler.value

def calc_delta_Qa(s_val, kf_val, hF_val, r_eff_val, log_width):
    """calc_delta_Qa berechnet Wasserandrang Qa mit Fehler."""
    Q_a = Qa_func(s_val, kf_val, hF_val, r_eff_val)
    dQ_dkf_val = dQ_dkf_func(s_val, kf_val, hF_val, r_eff_val)
    dQ_dhF_val = dQ_dhF_func(s_val, kf_val, hF_val, r_eff_val)
    # log-Raum
    sigma_log10 = log_width / math.sqrt(12)
    # Ableitung nach log10(kf)
    dQ_dlogkf = dQ_dkf_val * kf_val * math.log(10)
    delta_Qa = math.sqrt((dQ_dlogkf * sigma_log10)**2 + (dQ_dhF_val * delta_hF)**2)
    return float(Q_a), float(delta_Qa)

def calc_delta_Qf(s_val, kf_val, hF_val, r_bl_val, log_width):
    """calc_delta_Qaf berechnet das Fassungsvermögen Qf mit Fehler."""
    v_max = math.sqrt(kf_val) / 15
    Q_f = 2 * r_bl_val * math.pi * (hF_val - s_val) * v_max
    dQ_dkf = math.pi * r_bl_val * (hF_val - s_val) / (15 * math.sqrt(kf_val))
    dQ_dhF = 2 * math.pi * r_bl_val * math.sqrt(kf_val) / 15
    # log-Raum
    sigma_log10 = log_width / math.sqrt(12)
    # Ableitung nach log10(kf)
    dQ_dlogkf = dQ_dkf * kf_val * math.log(10)
    delta_Qf = math.sqrt((dQ_dlogkf * sigma_log10)**2 +(dQ_dhF * delta_hF)**2)
    return float(Q_f), float(delta_Qf)

In [ ]:
# Berechnung Qf, Qa und Unsicherheiten für s_array, in Dataframe df_Q
df_Q = pd.DataFrame({'s [m]': s_array})
df_Q[['Q_a [m3/s]', 'ΔQ_a [m3/s]']] = df_Q['s [m]'] \
    .apply(lambda s: calc_delta_Qa(s, kf_wert.value, hf_wert.value, calc_r_e, kf_fehler.value)) \
    .apply(pd.Series)

df_Q[['Q_f [m3/s]', 'ΔQ_f [m3/s]']] = df_Q['s [m]'] \
    .apply(lambda s: calc_delta_Qf(s, kf_wert.value, hf_wert.value, r_bl_wert.value, kf_fehler.value)) \
    .apply(pd.Series)

df_Q = df_Q[['s [m]', 'Q_f [m3/s]', 'ΔQ_f [m3/s]', 'Q_a [m3/s]', 'ΔQ_a [m3/s]']]

## 2.3. Optimale Brunnenleistung Qopt

In [ ]:
# Berechnung Qopt und sopt
x_s1 = s_array
y_Qf = df_Q["Q_f [m3/s]"]

x_s2 = s_array
y_Qa = df_Q["Q_a [m3/s]"]

# Datensätze interpolieren
f1 = interp1d(x_s1, y_Qf, kind='linear') # Qf
f2 = interp1d(x_s2, y_Qa, kind='linear') # Qa

# Definition der Differenzfunktion
def diff(x):
    return f1(x) - f2(x)

# Vorzeichenwechsel finden
xmin = max(min(x_s1), min(x_s2))
xmax = min(max(x_s1), max(x_s2))
xs = np.linspace(xmin, xmax, 100)
vals = diff(xs)
idx = np.where(np.sign(vals[:-1]) != np.sign(vals[1:]))[0]
schnittpunkt_x = []
schnittpunkt_y = []

for i in idx:
    a, b = xs[i], xs[i+1]       
    x_s = brentq(diff, a, b)    # Nullstelle
    y_s = float(f1(x_s))        # y-Wert 
    schnittpunkt_x.append(x_s)
    schnittpunkt_y.append(y_s)

output_Qopt = []

for i, (x, y) in enumerate(zip(schnittpunkt_x, schnittpunkt_y), 1):
    output_SP = widgets.HTML(f"""
    <div style="
        background: thistle;
        border-radius: 8px;
        padding: 10px 15px;
        margin: 8px 0;
        width: 300px;
        font-size:14px;
        line-height:1.5;">
        <b style="color:black">Der optimale Betriebspunkt befindet sich bei:</b><br>
        <span>s = {x:.2f} m</span><br>
        <span>Q = {y:.5f} m³/s</span>
    </div>
    """)

    output_Qopt.append(output_SP)

display(widgets.VBox(output_Qopt))

s_opt = float(schnittpunkt_x[0])
Q_opt = float(schnittpunkt_y[0])

# Plot Qopt
x = np.array(df_Q['s [m]'])
y_f = np.array(df_Q['Q_f [m3/s]'])
dy_f = np.array(df_Q['ΔQ_f [m3/s]'])
y_a = np.array(df_Q['Q_a [m3/s]'])
dy_a = np.array(df_Q['ΔQ_a [m3/s]'])

factor = 1000  # m3/s -> L/s
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)

axes[0].fill_between(x, y_f - dy_f, y_f + dy_f, color='blue', alpha=0.1)
axes[0].plot(x, y_f, color='blue', label='Q_f')
axes[0].fill_between(x, y_a - dy_a, y_a + dy_a, color='green', alpha=0.1)
axes[0].plot(x, y_a, color='green', label='Q_a')
axes[0].scatter(s_opt, Q_opt, color='purple', zorder=5, label="Optimaler Betriebspunkt")
axes[0].axvline(s_opt, color='purple', linestyle='--', alpha=0.7)
axes[0].axhline(Q_opt, color='purple', linestyle='--', alpha=0.7)
axes[0].annotate(f"s_opt = {s_opt:.2f} m\nQ_opt = {Q_opt:.5f} m³/s", (s_opt, Q_opt), textcoords="offset points", xytext=(5,20), color="purple")
axes[0].set_title("Qf und Qa in m³/s")
axes[0].set_xlabel("Absenkung s [m]")
axes[0].set_ylabel("Q_a und Q_f [m³/s]")
axes[0].set_ylim(0, 0.02)
axes[0].set_xlim(0,3*s_opt)
axes[0].legend()
axes[0].grid()
axes[1].remove()

plt.tight_layout()
plt.show()

In [ ]:
# Monte Carlo Qopt und sopt Berechnung
def calc_Qa_only(s_val, kf_val, hF_val, r_eff_val):
    """calc_Qa_only berechnet Qa (ohne Fehler) für eine gegebene Absenkung s, kf-Wert, Filterlänge hF und Brunnenradius r_eff."""
    return Qa_func(s_val, kf_val, hF_val, r_eff_val)

def calc_Qf_only(s_val, kf_val, hF_val, r_bl_val):
    """calc_Qf_only berechnet Qf (ohne Fehler) für eine gegebene Absenkung s, kf-Wert, Filterlänge hF und Bohrradius r_bl."""
    v_max = math.sqrt(kf_val) / 15
    Q_f = 2 * math.pi * r_bl_val * (hF_val - s_val) * v_max
    return float(Q_f)

def SP_function(kf_val, hF_val):
    """SP_function berechnet den Schnittpunkt von Qa und Qf für gegebene kf and hF Werte.
    Ausgabe der optimalen Absenkung s_opt und Förderrate Q_opt."""
    s_vals = np.linspace(0.01, hF_val*0.9, 500) # feine Wertereihe für Absenkung

    Qa_vals = []
    Qf_vals = []

    for s in s_vals:
        Qa = calc_Qa_only(s, kf_val, hF_val, calc_r_e)
        Qf = calc_Qf_only(s, kf_val, hF_val, r_bl_wert.value)
        Qa_vals.append(Qa)
        Qf_vals.append(Qf)

    diff = np.array(Qa_vals) - np.array(Qf_vals) # Differenz zwischen Qa und Qf für jedes s
    index = np.argmin(np.abs(diff)) # Index des minimalen Absolutwerts der Differenz

    return s_vals[index], Qa_vals[index]   # Ergebnis s_opt and Q_opt

def MCS(log_width, N=500):
    """MCS führt eine Monte Carlo Simulation durch zur Ermittlung des Mittelwerts und der Standardabweichung von s_opt und Q_opt.
    kf wird logarithmisch gesampelt innerhalb log_width,
    hF wird in einer Normalvertielung gesampelt."""

    Qopt_samples = []
    sopt_samples = []

    delta = log_width / 2
    log10_kf0 = np.log10(kf_wert.value)

    for i in range(N):
        # kf logarithmisch sampeln
        kf_rand = 10**np.random.uniform(log10_kf0 - delta, log10_kf0 + delta)

        # hF normal verteilen
        hF_rand = np.random.normal(hf_wert.value, hf_fehler.value)
        if hF_rand <= 0:
            continue

        s_opt, Q_opt = SP_function(kf_rand, hF_rand)

        sopt_samples.append(s_opt)
        Qopt_samples.append(Q_opt)

    return (np.mean(sopt_samples), np.std(sopt_samples), np.mean(Qopt_samples), np.std(Qopt_samples))

s_mean, s_std, Q_mean, Q_std = MCS(log_width=kf_fehler.value)

output_MCS = widgets.HTML(f"""
<div style="background: thistle; border-radius:8px;
padding:10px 15px; margin:8px 0; width:420px; font-size:14px;">

<b>Monte-Carlo Ergebnisse</b><br>

s = {s_mean:.2f} m (±{s_std:.2f} m)<br>
Qopt = {Q_mean:.5f} m³/s (±{Q_std:.5f} m³/s)<br><br>

</div>
""")
display(output_MCS)

# 3. Brunnendimensionierung

## 3.1. Minimaler Brunnenabstand

In [ ]:
# Berechnung des minimalen Brunnenabstands mit Fehler
kf = kf_wert.value # in m/s
sigma_log10 = kf_fehler.value / math.sqrt(12)

b = hf_wert.value # in m
delta_b = hf_fehler.value # Fehler b
i = i_wert.value
Q_vol = Q_vol_h_value[0]     

# d_min berechnen
d_min = (2*Q_vol)/(np.pi*kf*b*i)

# Ableitungen
dQ_dkf = (-2*Q_vol)/(math.pi*b*i*kf**2) 
dQ_db = (-2*Q_vol)/(math.pi*kf*i*b**2)
dQ_dlogkf = dQ_dkf * kf_wert.value * math.log(10)

# Fehlerfortpflanzung
delta_d_hyd = math.sqrt((dQ_dlogkf * sigma_log10)**2 + (dQ_db * delta_b)**2)


output_d_min = []
d_hyd_text = widgets.HTML(f"""
    <div style="
        background: honeydew;
        border-radius: 8px;
        padding: 10px 15px;
        margin: 8px 0;
        width: 300px;
        font-size:14px;
        line-height:1.5;">
        <style="color:black">Der minimale Brunnenabstand beträgt:<br>
        <span>{d_min:.1f} (±{delta_d_hyd:.1f} m)</span><br>
    </div>
    """)

output_d_min.append(d_hyd_text)

display(widgets.VBox(output_d_min))

## 3.2. Rückströmung

In [ ]:
# Berechnung Dimensionslose Förderrate nach Stauffer et al.
def Q_d_minimum(Q, b, d, k_f, i):
    a = d/2 #halber minimaler Abstand
    beta = round(Q / (np.pi * b * a * k_f * i),1)
    return beta

def Q_d_safe(Q, b, d, k_f, i):
    a = d # gewählter Abstand 2x größer als Mindestabstand
    beta = round(Q / (np.pi * b * a * k_f * i),1)
    return beta 

def Q_d_unsafe(Q, b, d, k_f, i):
    a = (d/2) * 0.5 # Hälfte des Mindesabstands
    beta = round(Q / (np.pi * b * a * k_f * i),1)
    return beta   

# Korrespondierend mit 3 Abstand-Szenarien (safe, minimum, unsafe)
good_beta = Q_d_safe(Q_vol_h_value[0], hf_wert.value, d_min, kf_wert.value, i_wert.value)
crit_beta = Q_d_minimum(Q_vol_h_value[0], hf_wert.value, d_min, kf_wert.value, i_wert.value)
bad_beta = Q_d_unsafe(Q_vol_h_value[0], hf_wert.value, d_min, kf_wert.value, i_wert.value)

# print(good_beta, crit_beta, bad_beta)

In [ ]:
def recirc_Q(beta):
    """recirc_Q berechnet das Verhältnis des vom Injektionsbrunnen zum Förderbrunnen zurück geströmten Grundwassers zu dem ungestörten Grundwasser.
    Args:
        beta (float): Dimensionslose Förderrate nach gewähltem Abstand
    Returns:
        float: Anteil der Rückstromrate I/Q
    """
    I_Q = (2/np.pi) * (-(1/beta)*np.sqrt(beta-1) + np.arctan(np.sqrt(beta-1)))
    return I_Q

calc_I_Q = round(recirc_Q(bad_beta),2)

# Veränderte GW-Temperatur am Förderbrunnen
Temp_neu_FBr = calc_I_Q * (GW_temp-T_slider.value) + (1-calc_I_Q)*GW_temp

# Veränderte Tempspreizung (nicht anwendbar, wenn deltaT = 1 K)
deltaT_neu = T_slider.value-1

# veränderte erforderliche Förderrate
Q_neu_h = calc_Q_vol(P_gw_h_max, deltaT_neu)

if "Kühlen" in betrieb.value:
    Q_neu_k = calc_Q_vol(P_gw_k_max, deltaT_neu)
    Q_neu = max(Q_neu_h[1], Q_neu_k[1]) # in m3/h
else:
    Q_neu = Q_neu_h[1]
    
output_rückstr = widgets.Output()
with output_rückstr:
    display(HTML(f"""
    <h3 style='color:black'>Rückströmung im Heizbetrieb</h3>
    <div style="
        background:honeydew;
        padding:15px;
        border-radius:12px;
        width:680px;
        font-size:14px;
        line-height:1.6">
            
        Der prozentuale Anteil der Förderrate, der bei einem Brunnenabstand von {round(d_min/2,2)} m aus zurückgeströmten Wasser stammt:</>
        <span style="color:black">
            {calc_I_Q* 100} %
        </span>
        <hr>
        Ungestörte Grundwasser-Temperatur:</>
        <span style="color:black">
            {round(GW_temp,2)} °C
        </span>
        <br>
        Temperaturveränderung des Förder-Grundwassers:</>
        <span style="color:black">
            {round(Temp_neu_FBr,2)} °C
        </span>
        <hr>
        Je größer die Temperaturveränderung am Förderbrunnen, desto größere Einschränkungen gibt es bei der Temperaturspreizung beim Wärmepumpenbetrieb.</>     
    </div>
    """))
display(output_rückstr)

# 4. Finale Auswertung

In [ ]:
Qopt_mean_m3h = Q_mean * 3600 

# Bewertung, ob erfordeliche Grundwasserförderrate gedeckt werden kann
if Q_vol_h_value[1] > Qopt_mean_m3h:
    GWL_msg = f"""
    <span style="color:black;">
    Mit einem Fördervolumen von <b>{round(Q_vol_h_value[1],2)} m³/h</b> besteht die Gefahr einer hydrogeologischen Überanspruchung des Aquifers.
    </span>
    """
else:
    GWL_msg = f"""
    <span style="color:black;">
    Das Fördervolumen von <b>{round(Q_vol_h_value[1],2)} m³/h</b> liegt unter dem optimalen Betriebspunkt.
    Eine Überanspruchung des Aquifers ist unwahrscheinlich.
    </span>
    """

output_zsmf_ergebnisse = f"""
<div style="
width:700px;
background-color:whitesmoke;
border-radius:14px;
padding:18px;
margin:8px 0;
font-size:14px;
line-height:1.6;
">

<h3 style="color:black; text-align:center;">
Zusammenfassung der Ergebnisse
</h3>

<hr style="border:none; border-top:1px solid #ddd;">
<p><b>Betriebsart:</b> 
{", ".join(betrieb.value) if isinstance(betrieb.value, (list, tuple)) else betrieb.value}
</p>
<p>
<b>Durchschnittliche Grundwassertemperatur:</b>
{round(GW_temp,1)} °C<br>
<b>Standort:</b> {Ort}
</p>
<p>
<b>Gebäudetyp:</b> {Geb_typ.value}<br>
<b>Gebäudefläche:</b> {Geb_fläche.value} m²
</p>
<p>
<b>Temperaturdifferenz des Grundwassers:</b>
{T_slider.value} K
</p>
<p>
<b>Maximal erforderliche GW-Förderrate (Heizbetrieb):</b>
<span style="color:cornflowerblue; font-size:14px;">
<b>{round(Q_vol_h_value[1],2)} m³/h</b>
</span>
</p>

<hr style="border:none; border-top:1px solid #ddd;">

<p>
<b>Hydrogeologische Eingangsparameter:</b><br>
<span style="color:black; font-size:12px;">
<b>kf-Wert:</b>
{round(kf_wert.value,5)} m/s ± Größenordnung {round(kf_fehler.value,2)}<br>
<b>Filterlänge hf:</b>
{round(hf_wert.value,2)} ± {round(hf_fehler.value,2)} m<br>
<b>Hydraulischer Gradient i:</b>
{round(i_wert.value,6)}<br>
<b>Brunnendurchmesser:</b>
{round(d_br_dropdown.value, 2)} mm
</span>
</p>

<hr style="border:none; border-top:1px solid #ddd;">

<p>
<b>Optimaler Betriebspunkt:</b><br>
<span style="color:purple; font-size:14px;">
<b>Die Optimale Förderrate beträgt {round(Qopt_mean_m3h,2)} (±{round(Q_std * 3600,2)}) m³/h
bei einer Absenkung von {round(s_mean,2)} (±{round(s_std,2)}) m.</b>
</span>
</p>
<p>{GWL_msg}</p>

<hr style="border:none; border-top:1px solid #ddd;">

<p>
<b>Berechnung des Brunnenabstands:</b><br>
<span style="color:black; font-size:14px;">
<b>Minimaler Brunnenabstand:</b>
<b>{round(d_min, 2)} m (± {round(delta_d_hyd, 2)} m)</b><br>
<span style="color:green; font-size:14px;">
<b>Sicherer Brunnenabstand:</b>
<b>{round(d_min*2, 2)} m (± {round(delta_d_hyd*2, 2)} m)</b><br>
<span style="color:red; font-size:14px;">
<b>Unsicherer Brunnenabstand:</b>
<b>{round(d_min*0.5, 2)} m (± {round(delta_d_hyd*0.5, 2)} m)</b><br>
<span style="color:black; font-size:12px;">
Bei dem unsicheren Abstand entsteht eine Rückströmung zwischen Förder- und Injektionsbrunnen, 
die zu einer Verringerung der Temperatur des geförderten Grundwassers auf {round(Temp_neu_FBr, 2)} °C führt (bei Heizbetrieb).<br>
Verringert sich so die erlaubte Temperaturspreizung des Wärmepumpenbetriebs auf {deltaT_neu} K, so erhöht sich die erforderliche Förderrate auf {round(Q_neu,2)} m³/h.
</span>
</p>
</div>
"""
display(HTML(output_zsmf_ergebnisse))
